# PokerMind AI — Modélisation de base (Notebook 02)

## 1. Présentation du notebook

### Objectif

Ce notebook entraîne et compare trois modèles de classification supervisés pour prédire si un joueur remporte une main de poker (`player_won = 1`). L'objectif est d'établir une première base de référence (*baseline*) solide et honnête sur le plan méthodologique.

### Données utilisées

Le jeu de données provient de mains No-Limit Texas Hold'em issues du dataset **Pluribus** — un agent IA développé par Facebook Research. Chaque ligne représente un joueur dans une main. La variable cible est `player_won` : 1 si le joueur a terminé avec plus de jetons qu'au départ, 0 sinon.

### Deux jeux de variables distincts

Ce notebook distingue deux jeux de variables pour mettre en évidence un problème de **fuite d'information** (*data leakage*) :

| Jeu de variables | Contenu | Timing |
|---|---|---|
| **Préflop** | Cartes initiales, position, blindes, taille de pile | Avant les décisions |
| **Main complète** | Variables préflop + comptes d'actions (plis, mises, relances…) | Après les décisions |

Cette distinction est cruciale : un modèle entraîné sur les variables d'action apprend à partir d'informations qui révèlent directement l'issue de la main, et ne constitue pas une prédiction honnête.

### Plan du notebook

1. Chargement et exploration initiale du jeu de données
2. Analyse du déséquilibre de classes
3. Définition des deux jeux de variables
4. Entraînement de trois modèles avec pondération des classes (`class_weight='balanced'`)
5. Validation croisée stratifiée à 5 plis pour des résultats robustes
6. Analyse de l'importance des variables — modèle préflop
7. Mise en évidence de la fuite d'information dans le modèle main complète
8. Bilan des limites et prochaine étape

## 2. Imports et configuration

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import cross_validate, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path("..")
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
PLAYER_FEATURE_FILE = PROCESSED_DATA_DIR / "player_level_features.csv"

pd.DataFrame(
    {
        "name": ["PROJECT_ROOT", "RAW_DATA_DIR", "PROCESSED_DATA_DIR", "PLAYER_FEATURE_FILE"],
        "path": [PROJECT_ROOT, RAW_DATA_DIR, PROCESSED_DATA_DIR, PLAYER_FEATURE_FILE],
    }
)

Les chemins affichés ci-dessus confirment la structure du projet. Si le fichier `player_level_features.csv` n'existe pas encore, il sera généré automatiquement dans la section suivante à partir des fichiers bruts PHH.

## 3. Chargement des données joueur

Le fichier `player_level_features.csv` est chargé depuis `data/processed/`. S'il n'existe pas encore, il est généré automatiquement via `src/player_features.py`. Chaque ligne représente un joueur dans une main.

In [ ]:
print(f"Répertoire de travail actuel : {Path('.').resolve()}")
print(f"Chemin absolu du fichier attendu : {PLAYER_FEATURE_FILE.resolve()}")

if not PLAYER_FEATURE_FILE.exists() or PLAYER_FEATURE_FILE.stat().st_size < 1024:
    sys.path.append(str(PROJECT_ROOT.resolve()))
    from src.player_features import save_player_level_features

    df, output_path = save_player_level_features(
        output_path=str(PLAYER_FEATURE_FILE),
        data_dir=str(RAW_DATA_DIR / "pluribus"),
        max_files=1000,
    )
    print(f"Jeu de données régénéré : {output_path}")
else:
    df = pd.read_csv(PLAYER_FEATURE_FILE)
    print(f"Jeu de données chargé depuis le fichier existant.")

required_columns = ["player_won", "preflop_strength_score", "starting_stack", "player_num_folds"]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Colonnes manquantes dans le jeu de données : {missing_columns}")

print(f"Forme du tableau : {df.shape[0]} lignes × {df.shape[1]} colonnes — OK")

## 4. Exploration initiale du jeu de données

On vérifie la structure du tableau : dimensions, aperçu des premières lignes, types de colonnes, valeurs manquantes, et distribution de la variable cible `player_won`.

**Ce qu'on cherche :** confirmer que le jeu de données est bien formé et détecter tout problème avant de modéliser.

In [ ]:
df.shape

In [ ]:
df.head(20)

In [ ]:
column_summary_df = pd.DataFrame(
    {
        "column_name": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "missing_values": df.isna().sum().values,
        "missing_percentage": (df.isna().mean().values * 100).round(2),
    }
)

column_summary_df

In [ ]:
df["player_won"].value_counts()

In [ ]:
df["profit"].describe()

### Observation — déséquilibre de classes

Le jeu de données présente un **déséquilibre important** :

- `player_won = 0` (perdant) : **4 999 lignes** (~83 %)
- `player_won = 1` (gagnant) : **1 001 lignes** (~17 %)

Ce déséquilibre est **naturel** : dans une partie à 6 joueurs, seul 1 joueur sur 6 gagne en moyenne. Il ne s'agit pas d'une erreur de données.

En revanche, il a des implications directes pour la modélisation :
- Un modèle qui prédit toujours `0` atteindrait ~83 % d'accuracy **sans rien apprendre**.
- L'**accuracy seule n'est pas une métrique fiable** dans ce contexte.
- On privilégiera le **recall**, le **F1-score** et l'**AUC-ROC**.
- Pour forcer les modèles à détecter les gagnants, on utilisera `class_weight='balanced'`.

## 5. Analyse exploratoire des variables préflop

On explore les distributions des principales variables disponibles avant les décisions pour comprendre leur relation avec la variable cible.

**Ce qu'on cherche :** identifier si certaines variables semblent corrélées avec la victoire, et repérer les distributions problématiques (constantes, uniformes, etc.).

In [ ]:
df["player_won"].value_counts().sort_index().plot(kind="bar", figsize=(6, 4))
plt.title("Target distribution: player_won")
plt.xlabel("player_won")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(data=df, x="player_won", y="preflop_strength_score")
plt.title("Distribution of preflop_strength_score by player_won")
plt.tight_layout()
plt.show()

In [ ]:
df["number_of_players"].value_counts().sort_index().plot(kind="bar", figsize=(6, 4))
plt.title("Distribution of number_of_players")
plt.xlabel("number_of_players")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
df.groupby("is_pair", dropna=False)["player_won"].mean().reset_index().plot(kind="bar", x="is_pair", y="player_won", legend=False, figsize=(6, 4))
plt.title("Average player_won by is_pair")
plt.xlabel("is_pair")
plt.ylabel("Average player_won")
plt.tight_layout()
plt.show()

In [ ]:
df.groupby("is_suited", dropna=False)["player_won"].mean().reset_index().plot(kind="bar", x="is_suited", y="player_won", legend=False, figsize=(6, 4))
plt.title("Average player_won by is_suited")
plt.xlabel("is_suited")
plt.ylabel("Average player_won")
plt.tight_layout()
plt.show()

In [ ]:
df.groupby("player_position_index", dropna=False)["player_won"].mean().reset_index().plot(kind="bar", x="player_position_index", y="player_won", legend=False, figsize=(8, 4))
plt.title("Average player_won by player_position_index")
plt.xlabel("player_position_index")
plt.ylabel("Average player_won")
plt.tight_layout()
plt.show()

### Observation — variables préflop et victoire

Les graphiques ci-dessus révèlent plusieurs points importants :

- **`preflop_strength_score`** : les gagnants ont tendance à avoir un score légèrement plus élevé, ce qui valide la cohérence de cet indicateur.
- **`number_of_players`** : toujours 6 dans ce jeu de données — Pluribus est une configuration fixe. Cette variable **n'apportera aucune information au modèle**.
- **`is_pair`** et **`is_suited`** : légère association positive avec la victoire, mais l'effet reste modeste.
- **`player_position_index`** : représente le numéro de siège (1–6), pas la position de jeu réelle. L'effet observé reflète donc la position de départ dans la rotation des blindes, pas la position stratégique (bouton, cutoff, etc.).

## 6. Séparation des jeux de variables : préflop et main complète

### Deux jeux de variables pour tester la fuite d'information

Certaines variables sont disponibles **avant les décisions** (préflop). D'autres n'existent que **pendant ou après les actions** de la main.

Si on utilise des variables d'action pour prédire l'issue, on risque une **fuite d'information** (*data leakage*) : le modèle apprend à partir d'informations qui, dans un cas réel, ne seraient disponibles qu'une fois la main terminée.

Pour mettre cela en évidence, on définit deux jeux de variables :

| Jeu de variables | Contenu | Timing |
|---|---|---|
| **`PREFLOP_FEATURES`** | Cartes initiales, position, blindes, taille de pile | Avant les décisions |
| **`FULL_HAND_FEATURES`** | Variables préflop + comptes d'actions (plis, mises, relances…) | Après les décisions |

On entraînera les mêmes modèles sur chaque jeu, puis on comparera les résultats. Si les performances sont radicalement meilleures sur le jeu complet, cela indiquera probablement une fuite d'information.

In [ ]:
target_column = "player_won"

PREFLOP_FEATURES = [
    "number_of_players",
    "starting_stack",
    "blind_or_straddle",
    "is_small_blind",
    "is_big_blind",
    "player_position_index",
    "is_pair",
    "is_suited",
    "high_card_rank",
    "low_card_rank",
    "rank_gap",
    "has_ace",
    "has_king",
    "preflop_strength_score",
    "variant",
]

FULL_HAND_FEATURES = PREFLOP_FEATURES + [
    "hand_action_count",
    "num_folds",
    "num_check_calls",
    "num_bets_raises",
    "num_hole_deals",
    "num_board_deals",
    "num_show_or_muck",
    "player_num_folds",
    "player_num_check_calls",
    "player_num_bets_raises",
    "player_num_show_or_muck",
]

target_y = df[target_column].copy()
stratify_value = target_y if target_y.nunique() > 1 else None

preflop_X = df[PREFLOP_FEATURES].copy()
full_hand_X = df[FULL_HAND_FEATURES].copy()

preflop_X_train, preflop_X_test, y_train, y_test = train_test_split(
    preflop_X,
    target_y,
    test_size=0.2,
    random_state=42,
    stratify=stratify_value,
)

full_hand_X_train, full_hand_X_test, _, _ = train_test_split(
    full_hand_X,
    target_y,
    test_size=0.2,
    random_state=42,
    stratify=stratify_value,
)

preflop_X_train.shape, preflop_X_test.shape, y_train.shape, y_test.shape

## 7. Entraînement et comparaison des modèles de base

On entraîne trois modèles supervisés classiques sur chaque jeu de variables :

- **Régression Logistique** (*Logistic Regression*) : modèle linéaire, interprétable, bon point de départ.
- **Arbre de Décision** (*Decision Tree*) : modèle non linéaire, facilement visualisable.
- **Forêt Aléatoire** (*Random Forest*) : ensemble de 200 arbres, généralement plus robuste.

**Paramètre clé — `class_weight='balanced'`**

Face au déséquilibre 83/17, tous les modèles sont entraînés avec `class_weight='balanced'`. Ce paramètre ajuste le poids de chaque classe en proportion inverse de sa fréquence, ce qui oblige le modèle à détecter les gagnants et non pas à prédire systématiquement "perdant".

Sans ce paramètre, un modèle qui prédit toujours 0 atteindrait 83 % d'accuracy sans rien apprendre.

In [ ]:
def split_feature_types(feature_list):
    numeric_feature_list = [feature for feature in feature_list if feature != "variant"]
    categorical_feature_list = [feature for feature in feature_list if feature == "variant"]
    return numeric_feature_list, categorical_feature_list


def train_and_compare_models(X_train, X_test, y_train, y_test, feature_list):
    numeric_feature_list, categorical_feature_list = split_feature_types(feature_list)

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessor = ColumnTransformer(transformers=[
        ("num", numeric_transformer, numeric_feature_list),
        ("cat", categorical_transformer, categorical_feature_list),
    ])

    model_definitions = {
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced"),
        "Decision Tree": DecisionTreeClassifier(random_state=42, class_weight="balanced"),
        "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"),
    }

    trained_model_map = {}
    comparison_rows = []
    confusion_matrix_map = {}

    for model_name, model in model_definitions.items():
        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ])
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        y_proba = pipeline.predict_proba(X_test)[:, 1]
        trained_model_map[model_name] = pipeline
        confusion_matrix_map[model_name] = confusion_matrix(y_test, y_pred)
        comparison_rows.append({
            "model_name": model_name,
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, y_proba),
            "train_accuracy": pipeline.score(X_train, y_train),
        })

    comparison_table = pd.DataFrame(comparison_rows).sort_values("roc_auc", ascending=False).reset_index(drop=True)
    return comparison_table, trained_model_map, confusion_matrix_map


preflop_comparison_df, preflop_trained_models, preflop_confusion_matrices = train_and_compare_models(
    preflop_X_train, preflop_X_test, y_train, y_test, PREFLOP_FEATURES
)
full_hand_comparison_df, full_hand_trained_models, full_hand_confusion_matrices = train_and_compare_models(
    full_hand_X_train, full_hand_X_test, y_train, y_test, FULL_HAND_FEATURES
)

### 7.1 Résultats — modèle préflop

Le modèle préflop utilise uniquement les informations disponibles avant les décisions. C'est le scénario de prédiction le plus honnête.

**Métriques à lire :** avec un déséquilibre 83/17, l'accuracy seule n'est pas fiable. On privilégie le **recall** (capacité à détecter les vrais gagnants), le **F1-score** (équilibre précision/recall), et surtout l'**AUC-ROC** (qualité globale de discrimination, indépendante du seuil de décision).

In [ ]:
preflop_comparison_df[["model_name", "accuracy", "precision", "recall", "f1", "roc_auc", "train_accuracy"]]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for axis, model_name in zip(axes, ["Logistic Regression", "Decision Tree", "Random Forest"]):
    sns.heatmap(preflop_confusion_matrices[model_name], annot=True, fmt="d", cmap="Blues", cbar=False, ax=axis)
    axis.set_title(f"Preflop: {model_name}")
    axis.set_xlabel("Predicted")
    axis.set_ylabel("Actual")

plt.tight_layout()
plt.show()

### Observation — résultats modèle préflop

Avec `class_weight='balanced'`, les modèles cherchent maintenant à détecter les gagnants, et non plus à prédire systématiquement "perdant".

L'**AUC-ROC** est la métrique principale ici : elle mesure la capacité du modèle à ordonner les joueurs par probabilité de victoire, indépendamment du seuil de décision. La colonne `train_accuracy` permet de détecter un éventuel surajustement (*overfitting*) si elle est nettement supérieure à l'accuracy de test.

Ces résultats constituent le **baseline préflop** : ce qu'un modèle peut apprendre uniquement à partir des informations disponibles avant les décisions.

### 7.2 Résultats — modèle main complète

Ce modèle inclut les variables d'action (nombre de plis, mises, relances…). Ses performances seront très probablement bien supérieures au modèle préflop — mais pour des raisons problématiques que l'analyse de la section 10 révélera en détail.

In [ ]:
full_hand_comparison_df[["model_name", "accuracy", "precision", "recall", "f1", "roc_auc", "train_accuracy"]]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for axis, model_name in zip(axes, ["Logistic Regression", "Decision Tree", "Random Forest"]):
    sns.heatmap(full_hand_confusion_matrices[model_name], annot=True, fmt="d", cmap="Blues", cbar=False, ax=axis)
    axis.set_title(f"Full-hand: {model_name}")
    axis.set_xlabel("Predicted")
    axis.set_ylabel("Actual")

plt.tight_layout()
plt.show()

### Observation — résultats modèle main complète

Les performances sont nettement supérieures au modèle préflop (accuracy > 97 %, AUC-ROC quasi parfait). Avant de conclure que ce modèle est excellent, il est essentiel de comprendre **pourquoi** ces résultats sont si élevés.

L'analyse de l'importance des variables en section 10 révélera que cette performance provient presque entièrement d'une seule variable d'action. Ce n'est pas un signe de bonne prédiction — c'est un symptôme de fuite d'information directe.

## 8. Validation croisée stratifiée (5 plis)

Un seul partage train/test peut produire des résultats variables selon la graine aléatoire choisie. Pour obtenir une estimation plus robuste et académiquement défendable, on applique une **validation croisée stratifiée à 5 plis** (*5-fold stratified cross-validation*).

La stratification garantit que chaque pli respecte les proportions de classes (83/17) du jeu de données complet, ce qui est essentiel avec un déséquilibre aussi marqué.

On applique la validation croisée uniquement sur le **modèle préflop**, qui est le seul scénario de prédiction honnête. Les résultats sont présentés sous forme de **moyenne ± écart-type** sur les 5 plis.

In [ ]:
cv_fold_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scoring_metrics = ["accuracy", "precision", "recall", "f1", "roc_auc"]

numeric_cv_features, categorical_cv_features = split_feature_types(PREFLOP_FEATURES)

preflop_models_for_cv = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"),
}

cv_results_rows = []
for model_name, model in preflop_models_for_cv.items():
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessor = ColumnTransformer(transformers=[
        ("num", numeric_transformer, numeric_cv_features),
        ("cat", categorical_transformer, categorical_cv_features),
    ])
    pipeline_for_cv = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model),
    ])
    scores = cross_validate(pipeline_for_cv, preflop_X, target_y, cv=cv_fold_strategy, scoring=cv_scoring_metrics)
    cv_results_rows.append({
        "modèle": model_name,
        "accuracy": f"{scores['test_accuracy'].mean():.3f} ± {scores['test_accuracy'].std():.3f}",
        "recall": f"{scores['test_recall'].mean():.3f} ± {scores['test_recall'].std():.3f}",
        "f1": f"{scores['test_f1'].mean():.3f} ± {scores['test_f1'].std():.3f}",
        "roc_auc": f"{scores['test_roc_auc'].mean():.3f} ± {scores['test_roc_auc'].std():.3f}",
    })

cv_results_df = pd.DataFrame(cv_results_rows)
cv_results_df

### Observation — résultats de la validation croisée

Le tableau ci-dessus présente les performances moyennes sur 5 plis, avec l'écart-type associé. Un faible écart-type indique que les résultats sont stables et ne dépendent pas d'une partition particulière.

**Métriques à retenir :**
- L'**AUC-ROC** est la métrique principale : elle mesure la capacité du modèle à classer un gagnant au-dessus d'un perdant, indépendamment du seuil de décision.
- Le **recall** mesure combien de vrais gagnants le modèle parvient à identifier.
- Ces estimations sur 5 plis sont plus fiables qu'un seul partage train/test.

C'est à partir de ces résultats que l'on choisit le **modèle préflop de référence** pour l'analyse approfondie du Notebook 03.

## 9. Importance des variables — modèle préflop

On analyse quelles variables ont le plus contribué aux prédictions du meilleur modèle préflop (par arbre de décision ou forêt aléatoire). Cela permet de comprendre ce que le modèle a réellement appris.

**Ce qu'on cherche :** identifier les variables les plus informatives, et repérer celles dont l'importance est nulle — ce qui révèle une limite propre au dataset Pluribus.

In [ ]:
preflop_tree_models = preflop_comparison_df[preflop_comparison_df["model_name"].isin(["Decision Tree", "Random Forest"])]
best_preflop_tree_model_name = preflop_tree_models.iloc[0]["model_name"]
best_preflop_pipeline = preflop_trained_models[best_preflop_tree_model_name]
best_preflop_preprocessor = best_preflop_pipeline.named_steps["preprocessor"]
best_preflop_model = best_preflop_pipeline.named_steps["model"]
preflop_feature_importance_df = pd.DataFrame(
    {
        "feature_name": best_preflop_preprocessor.get_feature_names_out(),
        "importance": best_preflop_model.feature_importances_,
    }
).sort_values("importance", ascending=False).head(15)

preflop_feature_importance_df

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=preflop_feature_importance_df, x="importance", y="feature_name", hue="feature_name", legend=False)
plt.title(f"Top 15 Feature Importances — {best_preflop_tree_model_name}")
plt.tight_layout()
plt.show()

### Observation — importance des variables préflop

Le graphique révèle deux catégories distinctes de variables :

**Variables qui contribuent le plus :**
- Les variables liées aux cartes initiales (`low_card_rank`, `preflop_strength_score`, `rank_gap`, `high_card_rank`) dominent. C'est cohérent avec le dataset Pluribus, où toutes les cartes sont visibles — la force de la main préflop est un signal réel et exploitable.
- `player_position_index` contribue modestement, reflétant que la position à table a un effet même avec un encodage imparfait (numéro de siège).

**Variables à importance nulle — limite du dataset Pluribus :**
- `starting_stack`, `number_of_players`, et `variant` ont une importance de **0.0**.
- La raison est structurelle : dans Pluribus, **tous les joueurs commencent avec 10 000 jetons**, toutes les parties ont exactement **6 joueurs**, et toutes les mains sont en variante **NT**.
- Ces variables ne portent aucune information discriminante dans ce contexte homogène.
- Dans un dataset de poker réel (tables différentes, tailles de pile variées, plusieurs variantes), elles pourraient être informatives.

Cette observation est une **limite importante de généralisation** : les conclusions de ce modèle sont spécifiques au format Pluribus.

## 10. Analyse de la fuite d'information — modèle main complète

Le modèle main complète affichait des performances bien supérieures. Cette section identifie précisément **pourquoi** et montre que ces résultats sont méthodologiquement trompeurs.

**Ce qu'on cherche :** identifier quelle variable est responsable de la performance apparente, et démontrer que le modèle n'apprend pas réellement à prédire le résultat d'une main.

In [ ]:
full_hand_tree_models = full_hand_comparison_df[full_hand_comparison_df["model_name"].isin(["Decision Tree", "Random Forest"])]
best_full_hand_tree_model_name = full_hand_tree_models.iloc[0]["model_name"]
best_full_hand_pipeline = full_hand_trained_models[best_full_hand_tree_model_name]
best_full_hand_preprocessor = best_full_hand_pipeline.named_steps["preprocessor"]
best_full_hand_model = best_full_hand_pipeline.named_steps["model"]
full_hand_feature_importance_df = pd.DataFrame(
    {
        "feature_name": best_full_hand_preprocessor.get_feature_names_out(),
        "importance": best_full_hand_model.feature_importances_,
    }
).sort_values("importance", ascending=False).head(15)

full_hand_feature_importance_df

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=full_hand_feature_importance_df, x="importance", y="feature_name", hue="feature_name", legend=False)
plt.title(f"Top 15 Feature Importances — {best_full_hand_tree_model_name}")
plt.tight_layout()
plt.show()

### Observation — fuite d'information directe via `player_num_folds`

Le graphique ci-dessus révèle un problème fondamental :

**`player_num_folds` représente à lui seul ~84 % de l'importance totale du modèle.**

---

#### Mécanisme exact de la fuite

`player_num_folds` est une variable binaire (0 ou 1). Elle vaut 1 si le joueur s'est couché pendant la main.

**Un joueur qui se couche (*fold*) ne peut jamais gagner le pot.** Sa valeur de `player_won` est donc mécaniquement contrainte à 0 dès que `player_num_folds = 1`.

> *A player who has folded is almost certain not to win the hand — fold-related features encode the result rather than predict it.*

Ce n'est pas une corrélation statistique apprise : c'est une tautologie encodée dans les données.

| `player_num_folds` | Conséquence sur `player_won` |
|---|---|
| 1 (le joueur s'est couché) | `player_won` est toujours 0 |
| 0 (le joueur n'a pas plié) | `player_won` peut être 0 ou 1 |

---

#### Problème chronologique

`player_num_folds` est calculée *après* que le fold a eu lieu — c'est-à-dire **après** (ou simultanément à) l'issue de la main pour ce joueur.

Elle est chronologiquement **en aval** du point de décision, pas en amont. Ce n'est pas une variable disponible au moment où une prédiction utile aurait lieu — c'est une variable qui *enregistre* ce que le joueur a déjà fait.

Inclure une telle variable dans un modèle prédictif constitue une **fuite d'information directe** (*data leakage*).

---

#### Ce que cela révèle sur le modèle main complète

Le modèle n'a pas appris à prédire qui gagne une main de poker. Il a appris à répondre à cette question :

> *Est-ce que ce joueur s'est couché ?*

Ce sont deux questions fondamentalement différentes. La première est une prédiction. La seconde est une détection post-hoc d'un événement déjà survenu.

Les 97 % d'accuracy ne reflètent aucune capacité prédictive réelle — ils mesurent une tautologie encodée dans les données.

---

**Conclusion :** seul le modèle préflop, qui n'utilise pas ces variables d'action, constitue un baseline honnête et académiquement défendable. C'est ce modèle qui est retenu comme référence principale. Le modèle main complète est conservé **uniquement comme outil diagnostique** pour illustrer l'ampleur et le mécanisme de la fuite.

## 11. Exploration non supervisée — K-Means

En complément de l'approche supervisée, on applique un clustering K-Means sur les variables préflop pour explorer si des groupes naturels de situations émergent dans les données.

**Important :** K-Means ne prédit pas `player_won`. C'est un outil d'exploration non supervisé pour visualiser la structure latente du jeu de données.

In [ ]:
kmeans_features = df[[feature for feature in PREFLOP_FEATURES if feature != "variant"]].copy()
kmeans_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
X_kmeans = kmeans_preprocessor.fit_transform(kmeans_features)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_kmeans = df.copy()
df_kmeans["cluster"] = kmeans.fit_predict(X_kmeans)

cluster_sizes_df = df_kmeans["cluster"].value_counts().sort_index().reset_index()
cluster_sizes_df.columns = ["cluster", "count"]
cluster_sizes_df

In [ ]:
cluster_summary_df = df_kmeans.groupby("cluster")[["player_won", "preflop_strength_score"]].mean().reset_index()
cluster_summary_df

### Observation — clusters K-Means

Les trois clusters révèlent une structure intéressante :

- Le cluster avec le **score préflop moyen le plus élevé** (~34) présente également le **taux de victoire moyen le plus fort** (~37 %).
- Les deux autres clusters ont des scores préflop similaires (~21) mais des taux de victoire différents, ce qui suggère que d'autres facteurs (position, blindes) influencent les groupes.

Cette exploration confirme que la force de la main préflop est corrélée avec la victoire, même en l'absence de supervision directe. C'est une validation qualitative cohérente avec les résultats du modèle supervisé.

## 12. Limites et conclusions

### Ce que le modèle préflop fait bien

- Il apprend un signal réel à partir des cartes initiales et de la position.
- Avec `class_weight='balanced'` et validation croisée stratifiée, les résultats sont méthodologiquement solides.
- L'AUC-ROC indique une capacité discriminante réelle, même si les performances absolues restent modestes — ce qui reflète la difficulté intrinsèque de la tâche.

### Limites identifiées

1. **Généralisation limitée à Pluribus.** Toutes les parties ont 6 joueurs avec des tailles de pile identiques (10 000). Les variables `starting_stack`, `number_of_players`, et `variant` n'apportent aucune information dans ce contexte spécifique.

2. **Position encodée comme numéro de siège.** `player_position_index` représente le siège du joueur (1–6), pas la position de jeu réelle (bouton, cutoff, blindes, etc.). La position réelle nécessiterait de parser l'information du bouton dans les actions PHH — cela n'a pas encore été implémenté.

3. **Proxy de victoire imparfait.** `player_won = 1` si `finishing_stack > starting_stack`. Ce proxy est simple et transparent, mais il ne mesure pas la qualité des décisions : un joueur peut perdre en jouant parfaitement, et gagner par chance.

4. **Pas d'équité théorique calculée.** Les probabilités produites par le modèle ne correspondent pas à des équités de solver. Ce sont des estimations statistiques issues des données historiques Pluribus.

5. **Fuite d'information dans le modèle main complète.** Démontré en section 10 : `player_num_folds` encode directement l'issue de la main. Ce modèle ne constitue pas un baseline valide et ne sera pas retenu.

6. **Cartes visibles dans Pluribus.** Dans ce dataset de recherche, toutes les cartes de tous les joueurs sont enregistrées. Dans un contexte réel de poker en ligne, seules les cartes du héros seraient disponibles — ce qui réduirait significativement la qualité des variables de carte.

### Conclusion

Le **modèle préflop avec pondération des classes** constitue le baseline le plus honnête pour cette étape du projet. Ses performances modestes reflètent la difficulté réelle de la tâche. Ces limites ne sont pas des erreurs — elles sont des observations méthodologiques importantes qui guideront les améliorations futures.

## 13. Prochaine étape — Notebook 03

Le **Notebook 03** approfondira l'analyse du modèle préflop retenu, sans entraîner de nouveaux modèles. Il se concentrera sur :

- **Courbe de calibration** : les probabilités produites par le modèle correspondent-elles à des fréquences observées réelles ?
- **Analyse des erreurs** : quels profils de joueurs le modèle prédit-il le plus mal, et pourquoi ?
- **Discussion de généralisation** : comment ce modèle se comporterait-il sur des données de poker réelles (cartes cachées, piles variables, formats différents) ?
- **Bilan consolidé des limites** : une liste complète et honnête des limites méthodologiques, formulée de façon académiquement défendable.

L'objectif du Notebook 03 est de **comprendre ce que le modèle a réellement appris**, et de formuler des conclusions rigoureuses à partir du baseline établi ici.